# Anti-Drone Defense SAC Training - Curriculum Learning
## Train on Easy, then transfer learn to Medium and Hard

This notebook implements **curriculum learning**: start with Easy environment, then fine-tune on progressively harder environments using transfer learning.

## 🔧 SETUP & INITIALIZATION

In [ ]:
# Core imports
import os
import sys
import subprocess
import logging
from pathlib import Path
from datetime import datetime
import json

# Setup logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s [%(levelname)s] %(message)s',
    datefmt='%H:%M:%S'
)
logger = logging.getLogger(__name__)

print("✓ Imports successful")
print(f"Python version: {sys.version.split()[0]}")

In [ ]:
# Define project structure
PROJECT_ROOT = Path(".")
ENV_DIR = PROJECT_ROOT / "ENV"
RESULTS_DIR = PROJECT_ROOT / "results"
MODELS_DIR = PROJECT_ROOT / "models"
CONFIGS_DIR = PROJECT_ROOT / "configs"

# Create directories if they don't exist
for directory in [ENV_DIR, RESULTS_DIR, MODELS_DIR, CONFIGS_DIR]:
    directory.mkdir(parents=True, exist_ok=True)
    logger.info(f"✓ Directory ready: {directory}")

# Environment paths
ENVIRONMENTS = {
    "easy": {
        "path": ENV_DIR / "Proti-dronova_Obrana_EASY" / "Proti-dronova_Obrana_EASY.exe",
        "name": "Easy",
        "color": "\033[92m",  # Green
    },
    "medium": {
        "path": ENV_DIR / "Proti-dronova_Obrana_MEDIUM" / "Proti-dronova_Obrana_MEDIUM.exe",
        "name": "Medium",
        "color": "\033[93m",  # Yellow
    },
    "hard": {
        "path": ENV_DIR / "Proti-dronova_Obrana_HARD" / "Proti-dronova_Obrana_HARD.exe",
        "name": "Hard",
        "color": "\033[91m",  # Red
    },
}

print("\n" + "="*60)
print("PROJECT STRUCTURE INITIALIZED")
print("="*60)

## 📋 ENVIRONMENT CHECK

In [ ]:
# Check which environments are available
available_envs = {}

print("\nEnvironment Status:")
print("-" * 60)

for difficulty, env_info in ENVIRONMENTS.items():
    path = env_info["path"]
    exists = path.exists()
    status = "✓" if exists else "✗"
    color = env_info["color"]
    reset_color = "\033[0m"
    
    print(f"{status} {color}{env_info['name']:8}{reset_color} - {path}")
    
    if exists:
        available_envs[difficulty] = path
        file_size = path.stat().st_size / (1024*1024)
        print(f"   Size: {file_size:.1f} MB\n")

print(f"\nFound: {len(available_envs)}/{len(ENVIRONMENTS)} environments")

if not available_envs:
    print("\n⚠️  WARNING: No environments found!")
    print("\nBuild instructions:")
    print("1. Open Easy scene in Unity")
    print("2. File → Build Settings → Build")
    print("3. Save to: ENV/Proti-dronova_Obrana_EASY/Proti-dronova_Obrana_EASY.exe")
    print("4. Repeat for Medium and Hard")
else:
    print("\n✓ Ready to train!")

## ⚙️ TRAINING CONFIGURATION

In [ ]:
# Curriculum learning stages
class CurriculumStage:
    def __init__(self, name, difficulty, steps, time_scale, learning_rate, transfer_from=None):
        self.name = name
        self.difficulty = difficulty
        self.steps = steps
        self.time_scale = time_scale
        self.learning_rate = learning_rate
        self.transfer_from = transfer_from
        self.run_id = f"turret_sac_{difficulty}_{name}"
        self.trained_model = None

# Define curriculum stages
curriculum_stages = [
    CurriculumStage(
        name="stage1",
        difficulty="easy",
        steps=500000,  # Shorter for initial training
        time_scale=20,
        learning_rate=0.0003,
        transfer_from=None
    ),
    CurriculumStage(
        name="stage2",
        difficulty="medium",
        steps=500000,  # Fine-tune
        time_scale=15,
        learning_rate=0.0001,  # Lower LR for fine-tuning
        transfer_from="easy"  # Transfer from Easy
    ),
    CurriculumStage(
        name="stage3",
        difficulty="hard",
        steps=500000,  # Final refinement
        time_scale=10,
        learning_rate=0.00005,  # Even lower LR for hard
        transfer_from="medium"  # Transfer from Medium
    ),
]

print("\nCurriculum Learning Plan:")
print("=" * 60)
for i, stage in enumerate(curriculum_stages, 1):
    print(f"\nStage {i}: {stage.name.upper()}")
    print(f"  Difficulty: {stage.difficulty.upper()}")
    print(f"  Steps: {stage.steps:,}")
    print(f"  Time Scale: {stage.time_scale}x")
    print(f"  Learning Rate: {stage.learning_rate}")
    if stage.transfer_from:
        print(f"  Transfer from: {stage.transfer_from.upper()}")
    print(f"  Run ID: {stage.run_id}")

In [ ]:
# SAC Hyperparameters (shared across stages)
SAC_HYPERPARAMS = {
    "batch_size": 256,
    "buffer_size": 1000000,
    "tau": 0.005,
    "init_entcoef": 1.0,
    "steps_per_update": 1,
    "reward_signal_steps_per_update": 1,
}

# Network architecture (shared)
NETWORK_CONFIG = {
    "normalize": True,
    "hidden_units": 256,
    "num_layers": 2,
    "vis_encode_type": "simple",
}

# Reward signals
REWARD_CONFIG = {
    "gamma": 0.99,
    "strength": 1.0,
}

print("✓ Training hyperparameters configured")

## 📝 YAML CONFIG GENERATOR

In [ ]:
def create_training_config(stage, learning_rate):
    """
    Create a YAML config file for a training stage
    """
    yaml_content = f"""behaviors:
  TurretBehavior:
    trainer_type: sac
    
    hyperparameters:
      batch_size: {SAC_HYPERPARAMS['batch_size']}
      buffer_size: {SAC_HYPERPARAMS['buffer_size']}
      learning_rate: {learning_rate}
      learning_rate_schedule: constant
      tau: {SAC_HYPERPARAMS['tau']}
      steps_per_update: {SAC_HYPERPARAMS['steps_per_update']}
      reward_signal_steps_per_update: {SAC_HYPERPARAMS['reward_signal_steps_per_update']}
      init_entcoef: {SAC_HYPERPARAMS['init_entcoef']}
      save_replay_buffer: false
    
    network_settings:
      normalize: {str(NETWORK_CONFIG['normalize']).lower()}
      hidden_units: {NETWORK_CONFIG['hidden_units']}
      num_layers: {NETWORK_CONFIG['num_layers']}
      vis_encode_type: {NETWORK_CONFIG['vis_encode_type']}
    
    reward_signals:
      extrinsic:
        gamma: {REWARD_CONFIG['gamma']}
        strength: {REWARD_CONFIG['strength']}
    
    keep_checkpoints: 5
    max_steps: {stage.steps}
    time_horizon: 64
    summary_freq: 10000
"""
    
    config_path = CONFIGS_DIR / f"{stage.run_id}.yaml"
    config_path.write_text(yaml_content)
    logger.info(f"✓ Created config: {config_path}")
    return config_path

# Generate configs for all stages
configs = {}
for stage in curriculum_stages:
    configs[stage.run_id] = create_training_config(stage, stage.learning_rate)

print(f"\n✓ Generated {len(configs)} configuration files")

## 🔍 MODEL FINDER

In [ ]:
def find_trained_model(run_id):
    """
    Find the trained ONNX model from a run
    """
    run_dir = RESULTS_DIR / run_id
    
    if not run_dir.exists():
        logger.warning(f"Run directory not found: {run_dir}")
        return None
    
    # Find all .onnx files
    model_files = list(run_dir.glob("**/TurretBehavior.onnx"))
    
    if not model_files:
        logger.warning(f"No .onnx model found in {run_dir}")
        return None
    
    # Get latest (most recent)
    latest_model = max(model_files, key=lambda p: p.stat().st_mtime)
    logger.info(f"✓ Found model: {latest_model}")
    return latest_model

def copy_model_for_transfer(from_run_id, to_run_id):
    """
    Copy trained model from one stage to use as checkpoint for next stage
    """
    model_path = find_trained_model(from_run_id)
    
    if not model_path:
        logger.error(f"Could not find model from {from_run_id}")
        return False
    
    # Prepare checkpoint directory
    checkpoint_dir = MODELS_DIR / to_run_id
    checkpoint_dir.mkdir(parents=True, exist_ok=True)
    
    # Copy model
    import shutil
    dest_path = checkpoint_dir / "TurretBehavior.onnx"
    shutil.copy(model_path, dest_path)
    logger.info(f"✓ Copied model for transfer learning: {dest_path}")
    return True

print("✓ Model finder functions ready")

## 🎯 TRAINING EXECUTOR

In [ ]:
def build_training_command(stage, config_path):
    """
    Build the mlagents-learn command for a stage
    """
    cmd = [
        "mlagents-learn",
        str(config_path),
        f"--run-id={stage.run_id}",
        f"--time-scale={stage.time_scale}",
        f"--width=1920",
        f"--height=1080",
    ]
    
    # Add environment if available (for headless training)
    if stage.difficulty in available_envs:
        env_path = available_envs[stage.difficulty]
        cmd.extend(["--env", str(env_path), "--no-graphics"])
    
    return cmd

def run_training_stage(stage, config_path):
    """
    Execute a single training stage
    """
    logger.info("\n" + "="*60)
    logger.info(f"STARTING: {stage.name.upper()} - {stage.difficulty.upper()}")
    logger.info("="*60)
    logger.info(f"Steps: {stage.steps:,}")
    logger.info(f"Time Scale: {stage.time_scale}x")
    logger.info(f"Learning Rate: {stage.learning_rate}")
    
    if stage.transfer_from:
        logger.info(f"Transfer Learning from: {stage.transfer_from.upper()}")
    logger.info("="*60 + "\n")
    
    # Build command
    cmd = build_training_command(stage, config_path)
    logger.info(f"Command: {' '.join(cmd)}")
    logger.info("\nWaiting for environment connection...")
    logger.info("💡 Start your environment now (built executable or Unity Editor play)\n")
    
    # Run training
    try:
        result = subprocess.run(cmd, check=False)
        if result.returncode == 0:
            logger.info(f"✓ {stage.name} completed successfully")
            return True
        else:
            logger.error(f"✗ {stage.name} failed with code {result.returncode}")
            return False
    except KeyboardInterrupt:
        logger.warning(f"⏹️  {stage.name} interrupted by user")
        return False
    except Exception as e:
        logger.error(f"✗ Error during {stage.name}: {e}")
        return False

print("✓ Training executor ready")

## 📊 TRAINING PROGRESS TRACKER

In [ ]:
def print_training_summary():
    """
    Print summary of training runs and models
    """
    print("\n" + "="*60)
    print("TRAINING SUMMARY")
    print("="*60)
    
    for stage in curriculum_stages:
        run_dir = RESULTS_DIR / stage.run_id
        exists = run_dir.exists()
        status = "✓" if exists else "✗"
        
        print(f"\n{status} {stage.name.upper()} ({stage.difficulty.upper()})")
        print(f"  Run ID: {stage.run_id}")
        
        if exists:
            model = find_trained_model(stage.run_id)
            if model:
                model_size = model.stat().st_size / (1024*1024)
                print(f"  ✓ Model: {model.name}")
                print(f"  Size: {model_size:.1f} MB")
            else:
                print(f"  ✗ Model not found (still training?)")
        else:
            print(f"  Not yet trained")
    
    print("\n" + "="*60)

print("✓ Tracker ready")

---

# 🚀 TRAINING EXECUTION

Choose your training path below:

## Option A: Train EASY only (foundation model)

In [ ]:
# Train only on EASY
stage = curriculum_stages[0]  # Easy stage
config = configs[stage.run_id]

# Check if environment available
if stage.difficulty not in available_envs:
    logger.error(f"Environment not available: {stage.difficulty}")
else:
    success = run_training_stage(stage, config)
    if success:
        logger.info("\n✓ Easy training complete!")
        model = find_trained_model(stage.run_id)
        if model:
            logger.info(f"Ready for transfer learning to Medium")

## Option B: Train EASY → MEDIUM (with transfer learning)

In [ ]:
# First check if we have Easy model, if not train it
easy_stage = curriculum_stages[0]
easy_model = find_trained_model(easy_stage.run_id)

if not easy_model:
    logger.info("Easy model not found, training Easy first...")
    run_training_stage(easy_stage, configs[easy_stage.run_id])

# Now train Medium with transfer learning
medium_stage = curriculum_stages[1]

if medium_stage.difficulty not in available_envs:
    logger.error(f"Environment not available: {medium_stage.difficulty}")
else:
    # Copy Easy model as checkpoint for Medium
    logger.info(f"\nPreparing transfer learning from {easy_stage.difficulty} to {medium_stage.difficulty}...")
    copy_model_for_transfer(easy_stage.run_id, medium_stage.run_id)
    
    # Train Medium
    success = run_training_stage(medium_stage, configs[medium_stage.run_id])
    if success:
        logger.info("\n✓ Medium training complete!")

## Option C: Full Curriculum (EASY → MEDIUM → HARD)

In [ ]:
# Run full curriculum learning pipeline
logger.info("\n🎓 STARTING FULL CURRICULUM LEARNING PIPELINE")
logger.info("Easy → Medium → Hard\n")

completed_stages = []

for stage in curriculum_stages:
    # Check if environment available
    if stage.difficulty not in available_envs:
        logger.error(f"✗ Environment not available: {stage.difficulty}")
        logger.info("Skipping this stage...\n")
        continue
    
    # Handle transfer learning
    if stage.transfer_from:
        logger.info(f"Preparing transfer from {stage.transfer_from}...")
        if not copy_model_for_transfer(
            [s.run_id for s in curriculum_stages if s.difficulty == stage.transfer_from][0],
            stage.run_id
        ):
            logger.warning(f"Transfer preparation failed, continuing with random init...")
    
    # Run training stage
    config = configs[stage.run_id]
    success = run_training_stage(stage, config)
    
    if success:
        completed_stages.append(stage.name)
        logger.info(f"✓ {stage.name} completed")
    else:
        logger.error(f"✗ {stage.name} failed")
        break  # Stop pipeline on failure

# Summary
logger.info(f"\n{'='*60}")
logger.info(f"PIPELINE COMPLETE")
logger.info(f"Completed stages: {completed_stages}")
logger.info(f"{'='*60}")

## 📊 View Training Results

In [ ]:
print_training_summary()

## 📦 Export Final Model

In [ ]:
def export_final_model(stage_name="stage3"):
    """
    Export the final trained model for use in Unity
    """
    stage = [s for s in curriculum_stages if s.name == stage_name][0]
    model_path = find_trained_model(stage.run_id)
    
    if not model_path:
        logger.error(f"Model not found for {stage_name}")
        return None
    
    # Create export directory
    export_dir = PROJECT_ROOT / "unity_models"
    export_dir.mkdir(parents=True, exist_ok=True)
    
    # Copy model
    import shutil
    export_path = export_dir / f"TurretBehavior_{stage_name}.onnx"
    shutil.copy(model_path, export_path)
    
    logger.info(f"\n✓ Model exported to: {export_path}")
    logger.info(f"\nImport into Unity:")
    logger.info(f"1. Copy to: Assets/Models/")
    logger.info(f"2. Set Behavior Type to 'Inference Only'")
    logger.info(f"3. Assign this .onnx model to your agent")
    
    return export_path

# Export the hardest model (most trained)
final_model = export_final_model("stage3")

if not final_model:
    logger.info("No hard model available, trying medium...")
    final_model = export_final_model("stage2")

if not final_model:
    logger.info("No medium model available, trying easy...")
    final_model = export_final_model("stage1")

## 📝 Training Log

In [2]:
def generate_training_report():
    """
    Generate a training report with all details
    """
    report = f"""\n{'='*60}
TRAINING REPORT - {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}
{'='*60}

CURRICULUM STAGES:
"""
    
    for i, stage in enumerate(curriculum_stages, 1):
        report += f"\nStage {i}: {stage.name.upper()}
"
        report += f"  Difficulty: {stage.difficulty.upper()}\n"
        report += f"  Steps: {stage.steps:,}\n"
        report += f"  Learning Rate: {stage.learning_rate}\n"
        report += f"  Time Scale: {stage.time_scale}x\n"
        
        if stage.transfer_from:
            report += f"  Transfer from: {stage.transfer_from.upper()}\n"
        
        model_path = find_trained_model(stage.run_id)
        if model_path:
            report += f"  ✓ Model: {model_path}\n"
        else:
            report += f"  ✗ No model found\n"
    
    report += f"\n{'='*60}\n"
    
    return report

report = generate_training_report()
print(report)

# Save report
report_path = PROJECT_ROOT / "training_report.txt"
report_path.write_text(report)
logger.info(f"Report saved to: {report_path}")

IndentationError: unindent does not match any outer indentation level (<tokenize>, line 29)